# Q1. 은행 계좌 만들기

In [ ]:
import random


class Account:
    bank_name = "SC은행"
    account_count = 0

    def __init__(self, owner: str, balance: int):
        self.owner = owner
        self.balance = balance

        self.account_number = self._generate_account_number()

        self.deposit_count = 0
        self._deposit_history = []
        self._withdraw_history = []

        Account.account_count += 1

    @staticmethod
    def _generate_account_number() -> str:
        part1 = str(random.randint(0, 999)).zfill(3)
        part2 = str(random.randint(0, 99)).zfill(2)
        part3 = str(random.randint(0, 999999)).zfill(6)
        return f"{part1}-{part2}-{part3}"

    @classmethod
    def get_account_num(cls) -> int:
        return cls.account_count

    def deposit(self, amount: int) -> None:
        if amount < 1:
            print("입금은 최소 1원 이상만 가능합니다.")
            return

        self.balance += amount
        self.deposit_count += 1
        self._deposit_history.append(amount)

        if self.deposit_count % 5 == 0:
            interest = int(self.balance * 0.01)
            self.balance += interest

    def withdraw(self, amount: int) -> None:
        if amount < 1:
            print("출금은 최소 1원 이상만 가능합니다.")
            return

        if amount > self.balance:
            print("잔고가 부족하여 출금할 수 없습니다.")
            return

        self.balance -= amount
        self._withdraw_history.append(amount)

    def display_info(self) -> None:
        print(
            f"은행이름: {Account.bank_name}, "
            f"예금주: {self.owner}, "
            f"계좌번호: {self.account_number}, "
            f"잔고: {self.balance:,}원"
        )

    def deposit_history(self) -> None:
        print(f"[{self.owner}] 입금 내역")
        if not self._deposit_history:
            print("내역 없음")
            return
        for i, amt in enumerate(self._deposit_history, start=1):
            print(f"{i}회: {amt:,}원")

    def withdraw_history(self) -> None:
        print(f"[{self.owner}] 출금 내역")
        if not self._withdraw_history:
            print("내역 없음")
            return
        for i, amt in enumerate(self._withdraw_history, start=1):
            print(f"{i}회: {amt:,}원")

In [ ]:
accounts = [
    Account("파이썬", 10000),
    Account("자바", 2000000),
    Account("C언어", 500000),
]

print("생성된 계좌 수:", Account.get_account_num())

# 입금 5회 테스트(이자 1% 적용 확인용)
accounts[0].deposit(1000)
accounts[0].deposit(1000)
accounts[0].deposit(1000)
accounts[0].deposit(1000)
accounts[0].deposit(1000)  # 5번째 입금 -> 잔고의 1% 이자 추가

accounts[0].withdraw(3000)

print("\n[잔고 100만원 이상 고객만 출력]")
for acc in accounts:
    if acc.balance >= 1_000_000:
        acc.display_info()

print("\n[입출금 내역 출력 예시]")
accounts[0].deposit_history()
accounts[0].withdraw_history()

생성된 계좌 수: 3

[잔고 100만원 이상 고객만 출력]
은행이름: SC은행, 예금주: 자바, 계좌번호: 566-97-239596, 잔고: 2,000,000원

[입출금 내역 출력 예시]
[파이썬] 입금 내역
1회: 1,000원
2회: 1,000원
3회: 1,000원
4회: 1,000원
5회: 1,000원
[파이썬] 출금 내역
1회: 3,000원


# Q2. 간단한 자동사냥 RPG 만들기

In [ ]:
import random

# Q1. Character 클래스 만들기
class Character:
    def __init__(self, name, level, hp, attack, defense):
        self.name = name
        self.level = level
        self.hp = hp
        self.attack = attack
        self.defense = defense

    # 인스턴스의 현재 체력이 0 초과인지 확인하여 bool 값 반환
    def is_alive(self):
        return self.hp > 0

    # 공격을 받았을 때 (받은 데미지 - 본인의 방어력)만큼 체력 감소
    def take_damage(self, damage):
        actual_damage = damage - self.defense
        # 방어력이 데미지보다 크면 체력이 감소하지 않음
        if actual_damage > 0:
            self.hp -= actual_damage

    # 타겟에게 데미지를 입히는 메서드 (1부터 공격력 사이 랜덤 정수)
    def attack_target(self, target):
        damage = random.randint(1, self.attack)
        target.take_damage(damage)
        return damage


# Q2. Player 클래스와 Monster 클래스 만들기
class Player(Character):
    def __init__(self, name):
        # 레벨 1, 체력 100, 공격력 25, 방어력 5로 초기화
        super().__init__(name, 1, 100, 25, 5)
        self.exp = 0  # 경험치 속성 추가

    # 경험치 획득 메서드
    def gain_experience(self, exp_points):
        self.exp += exp_points

    # 레벨업 메서드
    def level_up(self):
        if self.exp >= 50:
            self.level += 1
            self.attack += 10
            self.defense += 5
            self.exp -= 50  # 레벨업 후 경험치 차감 (또는 0으로 초기화)
            print(f"\n레벨 업! {self.name}의 레벨이 {self.level}이(가) 되었습니다.")
            print(f"공격력: {self.attack}, 방어력: {self.defense}")


class Monster(Character):
    def __init__(self, name, level):
        # 몬스터 레벨에 비례하는 체력, 공격력, 방어력 초기화
        hp = random.randint(10, 30) * level
        attack = random.randint(5, 20) * level
        defense = random.randint(1, 5) * level
        super().__init__(name, level, hp, attack, defense)


# Q3. battle 함수 만들기
def battle(player, monster):
    print(f"\n{monster.name} 과의 전투를 시작합니다.")

    # 둘 중 하나의 체력이 0 이하가 될 때까지 반복
    while player.is_alive() and monster.is_alive():
        # 플레이어 -> 몬스터 공격
        p_damage = player.attack_target(monster)
        print(f"{player.name}이(가) {monster.name}에게 {p_damage} 만큼 공격했다...!")
        print(f"{monster.name}의 체력: {monster.hp}")

        # 몬스터가 죽었으면 공격 불가 (반복문 탈출)
        if not monster.is_alive():
            break

        # 몬스터 -> 플레이어 공격
        m_damage = monster.attack_target(player)
        print(f"{monster.name}이(가) {player.name}에게 {m_damage} 만큼 공격했다...!")
        print(f"{player.name}의 체력: {player.hp}")

    # 전투 결과 처리
    if player.is_alive():
        print("전투 승리!")
        print(f"{monster.name} 을(를) 이겼다!")
        # 승리 보상: 몬스터 레벨 * 20 경험치
        player.gain_experience(monster.level * 20)
        player.level_up()
        return True
    else:
        print("전투 패배..")
        return False


# Q4. main 함수 만들기
def main():
    # 몬스터의 이름과 레벨이 매핑된 딕셔너리
    monster_dict = {'슬라임': 1, '고블린': 2, '오크': 3}

    # 플레이어 이름 입력 및 인스턴스 생성
    player_name = input("플레이어의 이름을 입력하세요: ")
    player = Player(player_name)

    # 몬스터 딕셔너리를 순회하며 전투 진행
    for m_name, m_level in monster_dict.items():
        monster = Monster(m_name, m_level)
        result = battle(player, monster)

        # 플레이어가 사망(패배)한 경우
        if not result:
            print("\n게임오버")
            break

    # 3마리를 모두 쓰러뜨리고 생존한 경우
    if player.is_alive():
        print("\n모든 몬스터를 물리쳤습니다! 게임 클리어!")

# 스크립트 실행
if __name__ == "__main__":
    main()


플레이어의 이름을 입력하세요: 임성배

슬라임 과의 전투를 시작합니다.
임성배이(가) 슬라임에게 8 만큼 공격했다...!
슬라임의 체력: 22
슬라임이(가) 임성배에게 11 만큼 공격했다...!
임성배의 체력: 94
임성배이(가) 슬라임에게 3 만큼 공격했다...!
슬라임의 체력: 21
슬라임이(가) 임성배에게 6 만큼 공격했다...!
임성배의 체력: 93
임성배이(가) 슬라임에게 19 만큼 공격했다...!
슬라임의 체력: 4
슬라임이(가) 임성배에게 14 만큼 공격했다...!
임성배의 체력: 84
임성배이(가) 슬라임에게 20 만큼 공격했다...!
슬라임의 체력: -14
전투 승리!
슬라임 을(를) 이겼다!

고블린 과의 전투를 시작합니다.
임성배이(가) 고블린에게 14 만큼 공격했다...!
고블린의 체력: 34
고블린이(가) 임성배에게 26 만큼 공격했다...!
임성배의 체력: 63
임성배이(가) 고블린에게 2 만큼 공격했다...!
고블린의 체력: 34
고블린이(가) 임성배에게 21 만큼 공격했다...!
임성배의 체력: 47
임성배이(가) 고블린에게 17 만큼 공격했다...!
고블린의 체력: 19
고블린이(가) 임성배에게 25 만큼 공격했다...!
임성배의 체력: 27
임성배이(가) 고블린에게 3 만큼 공격했다...!
고블린의 체력: 18
고블린이(가) 임성배에게 22 만큼 공격했다...!
임성배의 체력: 10
임성배이(가) 고블린에게 18 만큼 공격했다...!
고블린의 체력: 2
고블린이(가) 임성배에게 31 만큼 공격했다...!
임성배의 체력: -16
전투 패배..

게임오버
